In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate
from peft import LoraConfig, get_peft_model, TaskType
from collections import Counter
from transformers import BitsAndBytesConfig
from collections import Counter

from dotenv import load_dotenv
import os
load_dotenv()
YOUR_HF_TOKEN = os.getenv("YOUR_HF_TOKEN")

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def combine_text(row):
    # Collect all existing description parts in order
    desc_parts = [row.get(f"case_description_{i}", "") for i in range(1, 9) if row.get(f"case_description_{i}")]
    # Collect all existing justification parts in order
    just_parts = [row.get(f"justification_{i}", "") for i in range(1, 5) if row.get(f"justification_{i}")]
    
    # Prioritize description by putting it first
    return " ".join(desc_parts + just_parts)

In [3]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
            
        loss_fct = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # predictions = (logits > 0).astype(int)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
        "precision_macro": precision_score(labels, predictions, average="macro"),
        "recall_macro": recall_score(labels, predictions, average="macro")
    }

In [4]:
def plot_history(log_history):
    train_loss = []
    val_loss = []
    val_acc = []
    val_f1 = []
    
    for log in log_history:
        if "loss" in log and "eval_loss" not in log:
            train_loss.append(log["loss"])
    
        if "eval_loss" in log:
            val_loss.append(log["eval_loss"])
    
        if "eval_accuracy" in log:
            val_acc.append(log["eval_accuracy"])
    
        if "eval_f1_macro" in log:
            val_f1.append(log["eval_f1_macro"])
    
    
    # Convert to numpy arrays if needed
    train_loss = np.array(train_loss)
    val_loss = np.array(val_loss)
    val_acc = np.array(val_acc)
    val_f1 = np.array(val_f1)
    
    # X axes
    train_steps = np.arange(len(train_loss))
    val_steps = np.linspace(0, len(train_loss)-1, len(val_loss))
    
    # Create figure
    plt.figure(figsize=(12, 5))
    # plt.title("Supreme Court Decisions - JurBERT - 3 labels")
    
    # ----- Left subplot: Train / Val Loss -----
    plt.subplot(1, 2, 1)
    plt.plot(train_steps, train_loss, marker='o', label="Train Loss", linewidth=2)
    plt.plot(val_steps, val_loss, marker='o', label="Val Loss", linewidth=2)
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # ----- Right subplot: Val Acc / Val F1 -----
    plt.subplot(1, 2, 2)
    plt.plot(val_steps, val_acc, marker='o', label="Val Accuracy", linewidth=2)
    plt.plot(val_steps, val_f1, marker='o', label="Val F1 Macro", linewidth=2)
    plt.xlabel("Epochs")
    plt.ylabel("Score")
    plt.title("Validation Accuracy & F1")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
with open("data/supreme-court-data.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, list):
    labels = [item["label"] for item in data if "label" in item]
    
    label_counts = Counter(labels)
    
    print("Number of samples per label:")
    for label, count in sorted(label_counts.items()):
        print(f"{label:12} : {count:4d} samples")
    
    print(f"\nTotal samples in Supreme Court Data: {len(data)}")

Number of samples per label:
admis        :  396 samples
inadmisibil  :  588 samples
respins      :  416 samples

Total samples in Supreme Court Data: 1400


In [7]:
# Define your mapping
path = "data/supreme-court-data.json"
label_map = {
    "respins": 0,
    "admis": 1,
    "inadmisibil": 2,
}

def prepare_dataset(json_file_path, num_labels):
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if num_labels == 2:
        data = [item for item in data if item['label'] != 'inadmisibil']

    formatted_data = []
    for entry in data:
        # Combine description (1-8) and justification (1-4)
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()
        just = " ".join([entry.get(f"justification_{i}", "") for i in range(1, 5)]).strip()
        
        formatted_data.append({
            "text": f"{desc} {just}", # Description prioritized by order
            "label": label_map[entry["label"]]
        })
    
    # Split into train and validation (85/15)
    train_data, val_data = train_test_split(
        formatted_data, 
        test_size=0.15, 
        stratify=[d["label"] for d in formatted_data], # Keep class ratios same
        random_state=42
    )
    
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

# 3 labels task

In [8]:
train_raw, val_raw = prepare_dataset(path, 3)

## JurBERT

In [ ]:
model_name = "readerbench/jurBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        stride=256,
        return_overflowing_tokens=True,
        padding="max_length"
    )
    
    # Since one case can become 3-4 chunks, we must duplicate the label for each chunk
    sample_mapping = tokenized_inputs.pop("overflow_to_sample_mapping")
    labels = examples["label"]
    tokenized_inputs["labels"] = [labels[i] for i in sample_mapping]
    
    return tokenized_inputs

tokenized_train = train_raw.map(
    tokenize_and_align_labels, 
    batched=True, 
    remove_columns=train_raw.column_names
)

tokenized_val = val_raw.map(
    tokenize_and_align_labels, 
    batched=True, 
    remove_columns=val_raw.column_names
)

print(f"Original train cases: {len(train_raw)}")
print(f"Tokenized train windows: {len(tokenized_train)}")

The OrderedVocab you are attempting to save contains holes for indices [6, 7, 10, 41], your vocabulary could be corrupted!


Map: 100%|██████████| 1190/1190 [00:00<00:00, 1339.20 examples/s]


The OrderedVocab you are attempting to save contains holes for indices [6, 7, 10, 41], your vocabulary could be corrupted!


Map: 100%|██████████| 210/210 [00:00<00:00, 1361.85 examples/s]

Original train cases: 1190
Tokenized train windows: 1190


In [17]:
# compute label counts in train and validation sets and print with original label names
inv_label_map = {v: k for k, v in label_map.items()}

train_counts = Counter(train_raw["label"])
val_counts = Counter(val_raw["label"])

print(f"Train total: {len(train_raw)}")
for label_id, cnt in sorted(train_counts.items()):
    print(f"{inv_label_map.get(label_id, label_id):12} : {cnt}")

print(f"\nVal total:   {len(val_raw)}")
for label_id, cnt in sorted(val_counts.items()):
    print(f"{inv_label_map.get(label_id, label_id):12} : {cnt}")

Train total: 1190
respins      : 353
admis        : 337
inadmisibil  : 500

Val total:   210
respins      : 63
admis        : 59
inadmisibil  : 88


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=3
    ).to(device) 
print(f"Using device: {device}")

training_args = TrainingArguments(
    output_dir="results/supreme-jurbert-3",
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.15,
    per_device_train_batch_size=16,
    # gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.4,
    fp16=True,
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    metric_for_best_model="f1_macro", 
    greater_is_better=True,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train, # Your processed dataset
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
# trainer.train()

# log_history = trainer.state.log_history
# plot_history(log_history)

## RoBERT

In [ ]:
model_name = "readerbench/RoBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        stride=256,
        return_overflowing_tokens=True,
        padding="max_length"
    )
    
    # Since one case can become 3-4 chunks, we must duplicate the label for each chunk
    sample_mapping = tokenized_inputs.pop("overflow_to_sample_mapping")
    labels = examples["label"]
    tokenized_inputs["labels"] = [labels[i] for i in sample_mapping]
    
    return tokenized_inputs

tokenized_train = train_raw.map(
    tokenize_and_align_labels, 
    batched=True, 
    remove_columns=train_raw.column_names
)

tokenized_val = val_raw.map(
    tokenize_and_align_labels, 
    batched=True, 
    remove_columns=val_raw.column_names
)

print(f"Original train cases: {len(train_raw)}")
print(f"Tokenized train windows: {len(tokenized_train)}")

Map: 100%|██████████| 210/210 [00:00<00:00, 1276.59 examples/s]

Original train cases: 1190
Tokenized train windows: 1190


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=3
    ).to(device) 
print(f"Using device: {device}")

training_args = TrainingArguments(
    output_dir="results/supreme-robert-3",
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=16,
    # gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.2,
    fp16=True,
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    metric_for_best_model="f1_macro", 
    greater_is_better=True,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train, # Your processed dataset
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
# trainer.train()

# log_history = trainer.state.log_history
# plot_history(log_history)

## BERTBase

In [9]:
model_name = "dumitrescustefan/bert-base-romanian-cased-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # tokenizer.pad_token = tokenizer.eos_token
    # tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 210/210 [00:00<00:00, 2237.58 examples/s]


In [10]:
# def token_stats(dataset):
#     if "attention_mask" in dataset.column_names:
#         lengths = [int(sum(mask)) for mask in dataset["attention_mask"]]
#     else:
#         pad_id = tokenizer.pad_token_id
#         lengths = [int(np.sum(np.array(ids) != pad_id)) for ids in dataset["input_ids"]]
    
#     return {
#         "avg_tokens": float(np.mean(lengths)),
#         "max_tokens": int(np.max(lengths)),
#         "min_tokens": int(np.min(lengths)),
#     }

# train_stats = token_stats(tokenized_train)
# val_stats = token_stats(tokenized_val)

# print("tokenized_train stats:", train_stats)
# print("tokenized_val stats:", val_stats)

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=3
    ).to(device) 
print(f"Using device: {device}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dumitrescustefan/bert-base-romanian-cased-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


In [12]:
training_args = TrainingArguments(
    output_dir="results/good/supreme-robertbase-3-labels",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
50,1.040200,0.841810,0.800000,0.768923,0.804762,0.768878
100,0.668100,0.474043,0.857143,0.833680,0.863363,0.834164
150,0.523400,0.408752,0.857143,0.834380,0.858579,0.833019
200,0.447100,0.291354,0.914286,0.905530,0.917414,0.900099
250,0.438500,0.351757,0.909524,0.901452,0.910974,0.896242
300,0.326900,0.323012,0.895238,0.883178,0.892222,0.878576
350,0.278100,0.303789,0.900000,0.892102,0.898219,0.887949


TrainOutput(global_step=375, training_loss=0.5114369506835937, metrics={'train_runtime': 90.7258, 'train_samples_per_second': 65.582, 'train_steps_per_second': 4.133, 'total_flos': 1565524835481600.0, 'train_loss': 0.5114369506835937, 'epoch': 5.0})

## LegalBERT

In [12]:
model_name = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # tokenizer.pad_token = tokenizer.eos_token
    # tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 210/210 [00:00<00:00, 1656.04 examples/s]


In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=3
    ).to(device) 
print(f"Using device: {device}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cpu


In [14]:
training_args = TrainingArguments(
    output_dir="results/supreme-bert-llm",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
50,1.034200,0.844884,0.761905,0.731523,0.741399,0.733502
100,0.762900,0.606012,0.752381,0.716574,0.751648,0.728438
150,0.616400,0.453207,0.876190,0.861457,0.869286,0.859342
200,0.499000,0.405159,0.847619,0.829982,0.838277,0.827169
250,0.494800,0.610401,0.766667,0.740998,0.786672,0.751758
300,0.421900,0.402748,0.861905,0.839764,0.873444,0.840958
350,0.380000,0.399627,0.833333,0.814617,0.816940,0.813226


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory wo

TrainOutput(global_step=375, training_loss=0.5824475186665853, metrics={'train_runtime': 4469.005, 'train_samples_per_second': 1.331, 'train_steps_per_second': 0.084, 'total_flos': 1565524835481600.0, 'train_loss': 0.5824475186665853, 'epoch': 5.0})

# 2 labels task

In [13]:
train_raw, val_raw = prepare_dataset(path, 2)

## JurBERT

In [ ]:
model_name = "readerbench/jurBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        stride=256,
        return_overflowing_tokens=True,
        padding="max_length"
    )
    
    # Since one case can become 3-4 chunks, we must duplicate the label for each chunk
    sample_mapping = tokenized_inputs.pop("overflow_to_sample_mapping")
    labels = examples["label"]
    tokenized_inputs["labels"] = [labels[i] for i in sample_mapping]
    
    return tokenized_inputs

tokenized_train = train_raw.map(
    tokenize_and_align_labels, 
    batched=True, 
    remove_columns=train_raw.column_names
)

tokenized_val = val_raw.map(
    tokenize_and_align_labels, 
    batched=True, 
    remove_columns=val_raw.column_names
)

print(f"Original train cases: {len(train_raw)}")
print(f"Tokenized train windows: {len(tokenized_train)}")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
    ).to(device) 
print(f"Using device: {device}")

training_args = TrainingArguments(
    output_dir="results/supreme-jurbert-2",
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.15,
    per_device_train_batch_size=16,
    # gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.4,
    fp16=True,
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    metric_for_best_model="f1_macro", 
    greater_is_better=True,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train, # Your processed dataset
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
# trainer.train()

# log_history = trainer.state.log_history
# plot_history(log_history)

## RoBERT

In [ ]:
model_name = "readerbench/RoBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        stride=256,
        return_overflowing_tokens=True,
        padding="max_length"
    )
    
    # Since one case can become 3-4 chunks, we must duplicate the label for each chunk
    sample_mapping = tokenized_inputs.pop("overflow_to_sample_mapping")
    labels = examples["label"]
    tokenized_inputs["labels"] = [labels[i] for i in sample_mapping]
    
    return tokenized_inputs

tokenized_train = train_raw.map(
    tokenize_and_align_labels, 
    batched=True, 
    remove_columns=train_raw.column_names
)

tokenized_val = val_raw.map(
    tokenize_and_align_labels, 
    batched=True, 
    remove_columns=val_raw.column_names
)

print(f"Original train cases: {len(train_raw)}")
print(f"Tokenized train windows: {len(tokenized_train)}")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
    ).to(device) 
print(f"Using device: {device}")

training_args = TrainingArguments(
    output_dir="results/supreme-robert-2",
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=16,
    # gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.2,
    fp16=True,
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    metric_for_best_model="f1_macro", 
    greater_is_better=True,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train, # Your processed dataset
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
# trainer.train()

# log_history = trainer.state.log_history
# plot_history(log_history)

## BertBase

In [14]:
model_name = "dumitrescustefan/bert-base-romanian-cased-v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # tokenizer.pad_token = tokenizer.eos_token
    # tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 122/122 [00:00<00:00, 2999.14 examples/s]


In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
    ).to(device) 
print(f"Using device: {device}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dumitrescustefan/bert-base-romanian-cased-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


In [17]:
training_args = TrainingArguments(
    output_dir="results/good/supreme-robertbase-2-labels",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
50,0.129100,0.402276,0.827869,0.827579,0.834150,0.830105
100,0.083100,0.434576,0.901639,0.901401,0.902294,0.900995
150,0.057700,0.469386,0.885246,0.885215,0.887989,0.886737
200,0.044200,0.485346,0.909836,0.909782,0.909677,0.910008
250,0.076000,0.527677,0.893443,0.893378,0.893280,0.893597
300,0.064200,0.593628,0.893443,0.893263,0.893588,0.893059
350,0.017700,0.701823,0.893443,0.893263,0.893588,0.893059
400,0.004600,0.667422,0.901639,0.901533,0.901533,0.901533


TrainOutput(global_step=440, training_loss=0.05429874174296856, metrics={'train_runtime': 104.9594, 'train_samples_per_second': 65.74, 'train_steps_per_second': 4.192, 'total_flos': 1815466281984000.0, 'train_loss': 0.05429874174296856, 'epoch': 10.0})

## LegalBERT

In [12]:
model_name = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # tokenizer.pad_token = tokenizer.eos_token
    # tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 122/122 [00:00<00:00, 1866.70 examples/s]


In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
    ).to(device) 
print(f"Using device: {device}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


In [ ]:
# training_args = TrainingArguments(
#     output_dir="results/supreme-bert-llm",
#     learning_rate=2e-5,
#     lr_scheduler_type="cosine",
#     warmup_steps=200,
#     warmup_ratio=0.1,
#     per_device_train_batch_size=8,
#     gradient_accumulation_steps=2,
#     num_train_epochs=10,
#     weight_decay=0.1,
#     fp16=True,
#     logging_steps=50,
#     eval_steps=50,
#     eval_strategy="steps",
#     save_strategy="epoch",
#     report_to="none",
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_train,
#     eval_dataset=tokenized_val,
#     compute_metrics=compute_metrics,
#     data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
# )

# trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
50,0.704500,0.657527,0.663934,0.629893,0.730369,0.654695
100,0.610000,0.485295,0.770492,0.770430,0.770492,0.770783
150,0.490000,0.399122,0.811475,0.810444,0.825000,0.814770
200,0.384300,0.511220,0.754098,0.752435,0.767682,0.757600
250,0.394300,0.619445,0.745902,0.745062,0.753846,0.748588
300,0.247700,0.485378,0.811475,0.811158,0.811422,0.811003
350,0.129700,0.565162,0.819672,0.818452,0.823079,0.817864
400,0.100900,0.612193,0.836066,0.834957,0.839759,0.834275


TrainOutput(global_step=440, training_loss=0.3552085188302127, metrics={'train_runtime': 94.0477, 'train_samples_per_second': 73.367, 'train_steps_per_second': 4.678, 'total_flos': 1815466281984000.0, 'train_loss': 0.3552085188302127, 'epoch': 10.0})

# LLMs

In [ ]:
try:
    from huggingface_hub import login
    login(YOUR_HF_TOKEN)
except Exception as e:
    print("Hugging Face login skipped:", e)

In [10]:
def prepare_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    label_map = {"respins": 0, "admis": 1}
    data = [item for item in data if item['label'] != 'inadmisibil']

    formatted_data = []
    for entry in data:
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()
        
        # --- PROMPT ENGINEERING START ---
        
        just = " ".join([entry.get(f"justification_{i}", "") for i in range(1, 5)]).strip()
        instructional_prompt = (
            f"Decide verdictul acestui caz juridic:\n\n"
            f"DESCRIERE: {desc}\n"
            f"JUSTIFICARE: {just}\n\n"
            f"Ai de ales între două răspunsuri: admis sau respins.\n"
        )

        # --- PROMPT ENGINEERING END ---
        
        formatted_data.append({
            "text": instructional_prompt, 
            "label": label_map[entry["label"]]
        })
    
    train_data, val_data = train_test_split(
        formatted_data, 
        test_size=0.15, 
        stratify=[d["label"] for d in formatted_data], # Keep class ratios same
        random_state=42
    )
    
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

In [11]:
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    results = clf_metrics.compute(predictions=predictions, references=labels)
    
    decoded_preds = [str(p) for p in predictions]
    decoded_labels = [str(l) for l in labels]
    
    results.update(rouge.compute(predictions=decoded_preds, references=decoded_labels))    
    return results

In [12]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

## Llama

In [ ]:
model_name = "meta-llama/Llama-2-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


In [ ]:
device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

In [ ]:
# training_args = TrainingArguments(
#     output_dir="results/supreme-llama-llm",
#     lr_scheduler_type="reduce_lr_on_plateau",
#     learning_rate=5e-5,
#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=4,
#     num_train_epochs=3,
#     weight_decay=0.2,
#     gradient_checkpointing=True,       # Crucial for saving VRAM
#     optim="paged_adamw_32bit",         # Memory-efficient optimizer
#     logging_steps=150,
#     eval_steps=150,
#     eval_strategy="steps",
#     save_strategy="epoch",
#     report_to="none",
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_train,
#     eval_dataset=tokenized_val,
#     compute_metrics=compute_metrics,
#     data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
# )

# trainer.train()

## RoLlama

In [16]:
model_name = "OpenLLM-Ro/RoLlama2-7b-Base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 122/122 [00:00<00:00, 2930.83 examples/s]


In [17]:
device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

Loading checkpoint shards: 100%|██████████| 3/3 [00:09<00:00,  3.20s/it]
Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoLlama2-7b-Base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.2,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum
150,1.275600,1.158068,0.483607,0.577181,0.477778,0.728814,0.483607,0.000000,0.483607,0.483607
300,1.244900,1.433172,0.459016,0.587500,0.465347,0.796610,0.459016,0.000000,0.459016,0.459016
450,1.264200,1.140619,0.459016,0.507463,0.453333,0.576271,0.459016,0.000000,0.459016,0.459016
600,1.233500,1.128702,0.442623,0.413793,0.421053,0.406780,0.442623,0.000000,0.442623,0.442623
750,1.085900,1.140922,0.450820,0.385321,0.420000,0.355932,0.450820,0.000000,0.450820,0.450820
900,1.129500,1.295515,0.467213,0.525547,0.461538,0.610169,0.467213,0.000000,0.467213,0.467213
1050,0.991400,1.404971,0.434426,0.496350,0.435897,0.576271,0.434426,0.000000,0.434426,0.434426
1200,1.199000,1.289059,0.491803,0.500000,0.476923,0.525424,0.491803,0.000000,0.491803,0.491803
1350,1.061100,1.355040,0.442623,0.507246,0.443038,0.593220,0.442623,0.000000,0.442623,0.442623
1500,1.150100,1.343550,0.475410,0.475410,0.460317,0.491525,0.475410,0.000000,0.475410,0.475410


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

TrainOutput(global_step=2070, training_loss=1.1046180282813916, metrics={'train_runtime': 1332.9293, 'train_samples_per_second': 1.553, 'train_steps_per_second': 1.553, 'total_flos': 4.120964619042816e+16, 'train_loss': 1.1046180282813916, 'epoch': 3.0})

## RoGemma

In [13]:
model_name = "OpenLLM-Ro/RoGemma-7b-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 122/122 [00:00<00:00, 2486.82 examples/s]


In [14]:
device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

Loading checkpoint shards: 100%|██████████| 4/4 [00:10<00:00,  2.54s/it]
Some weights of GemmaForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoGemma-7b-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.2,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum
150,2.099800,1.772569,0.565574,0.485437,0.568182,0.423729,0.565574,0.000000,0.565574,0.565574
300,1.999500,1.535476,0.540984,0.548387,0.523077,0.576271,0.540984,0.000000,0.540984,0.540984
450,1.822700,1.913286,0.573770,0.380952,0.640000,0.271186,0.573770,0.000000,0.573770,0.573770
600,1.827800,1.535027,0.581967,0.474227,0.605263,0.389831,0.581967,0.000000,0.581967,0.581967
750,1.614600,2.083428,0.549180,0.303797,0.600000,0.203390,0.549180,0.000000,0.549180,0.549180
900,1.779100,1.981803,0.516393,0.646707,0.500000,0.915254,0.516393,0.000000,0.516393,0.516393
1050,1.406000,1.255050,0.663934,0.655462,0.650000,0.661017,0.663934,0.000000,0.663934,0.663934
1200,1.149100,1.231574,0.655738,0.637931,0.649123,0.627119,0.655738,0.000000,0.655738,0.655738
1350,1.106800,1.240295,0.647541,0.638655,0.633333,0.644068,0.647541,0.000000,0.647541,0.647541
1500,1.253000,1.538366,0.639344,0.531915,0.714286,0.423729,0.639344,0.000000,0.639344,0.639344


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

TrainOutput(global_step=2070, training_loss=1.5031612562096637, metrics={'train_runtime': 1545.2975, 'train_samples_per_second': 1.34, 'train_steps_per_second': 1.34, 'total_flos': 4.931100047572992e+16, 'train_loss': 1.5031612562096637, 'epoch': 3.0})

## RoMistral

In [13]:
model_name = "OpenLLM-Ro/RoMistral-7b-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 122/122 [00:00<00:00, 3024.33 examples/s]


In [14]:
device_map = {"": 0}

# Model Loading with LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    quantization_config=bnb_config, # Add this
    num_labels=2,
    dtype=torch.float32, # Ensure model weights are in float32 for LoRA
    device_map=device_map
)
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Target the attention layers
)

model = get_peft_model(model, peft_config)

Loading checkpoint shards: 100%|██████████| 3/3 [00:10<00:00,  3.34s/it]
Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at OpenLLM-Ro/RoMistral-7b-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
training_args = TrainingArguments(
    output_dir="results/supreme-llama-llm",
    lr_scheduler_type="reduce_lr_on_plateau",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.2,
    gradient_checkpointing=True,       # Crucial for saving VRAM
    optim="paged_adamw_32bit",         # Memory-efficient optimizer
    logging_steps=150,
    eval_steps=150,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum
150,3.571300,3.219237,0.459016,0.400000,0.431373,0.372881,0.459016,0.000000,0.459016,0.459016
300,3.279900,3.192946,0.467213,0.453782,0.450000,0.457627,0.467213,0.000000,0.467213,0.467213
450,3.680300,3.226410,0.467213,0.329897,0.421053,0.271186,0.467213,0.000000,0.467213,0.467213
600,3.342200,3.351190,0.483607,0.292135,0.433333,0.220339,0.483607,0.000000,0.483607,0.483607
750,2.333000,3.333273,0.500000,0.314607,0.466667,0.237288,0.500000,0.000000,0.500000,0.500000
900,2.217000,3.141451,0.549180,0.630872,0.522222,0.796610,0.549180,0.000000,0.549180,0.549180
1050,2.666900,2.699496,0.516393,0.486957,0.500000,0.474576,0.516393,0.000000,0.516393,0.516393
1200,2.630700,2.747663,0.516393,0.458716,0.500000,0.423729,0.516393,0.000000,0.516393,0.524590
1350,2.202600,2.739369,0.524590,0.472727,0.509804,0.440678,0.524590,0.000000,0.516393,0.524590
1500,1.952500,2.906587,0.500000,0.371134,0.473684,0.305085,0.500000,0.000000,0.500000,0.500000


/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:232: UserWarning:

TrainOutput(global_step=2070, training_loss=2.5264740874801856, metrics={'train_runtime': 1493.0322, 'train_samples_per_second': 1.386, 'train_steps_per_second': 1.386, 'total_flos': 4.440525486686208e+16, 'train_loss': 2.5264740874801856, 'epoch': 3.0})

## RoGPT2

In [27]:
model_name = "readerbench/RoGPT2-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

tokenized_train = train_raw.map(tokenize_function, batched=True)
tokenized_val = val_raw.map(tokenize_function, batched=True)


Map: 100%|██████████| 122/122 [00:00<00:00, 2528.26 examples/s]


In [28]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Important for GPT2 padding
model.config.pad_token_id = tokenizer.pad_token_id

model = model.to(device)

print(f"Using device: {device}")

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at readerbench/RoGPT2-base and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


In [29]:
training_args = TrainingArguments(
    output_dir="results/supreme-bert-llm",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    weight_decay=0.1,
    fp16=True,
    logging_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Rouge1,Rouge2,Rougel,Rougelsum
50,0.753300,0.757978,0.500000,0.460177,0.481481,0.440678,0.500000,0.000000,0.500000,0.500000
100,0.684000,0.664631,0.581967,0.556522,0.571429,0.542373,0.581967,0.000000,0.581967,0.581967
150,0.542600,0.573587,0.672131,0.649123,0.672727,0.627119,0.672131,0.000000,0.672131,0.672131
200,0.372000,0.486996,0.770492,0.774194,0.738462,0.813559,0.770492,0.000000,0.770492,0.770492
250,0.196200,0.871960,0.713115,0.758621,0.639535,0.932203,0.713115,0.000000,0.713115,0.713115
300,0.137100,0.606693,0.795082,0.812030,0.729730,0.915254,0.795082,0.000000,0.795082,0.795082
350,0.047700,0.552742,0.786885,0.790323,0.753846,0.830508,0.786885,0.000000,0.786885,0.786885
400,0.023200,0.574425,0.778689,0.784000,0.742424,0.830508,0.778689,0.000000,0.778689,0.770492


TrainOutput(global_step=440, training_loss=0.31483437662774866, metrics={'train_runtime': 155.9204, 'train_samples_per_second': 44.253, 'train_steps_per_second': 2.822, 'total_flos': 1802947579084800.0, 'train_loss': 0.31483437662774866, 'epoch': 10.0})